# Chunk 51 — Balanced single-band datasets for 12×12, 15×15, 35×35 and 45×45 (data preparation only)

**Goal:** a controlled study in which every non-25×25 grid has a training/val/test set of the **same size (5,000)**, with **about 50% single-band and 50% non-functioning** antennas. The GATv2 fine-tune and the Gupta CNN are then both trained on these sets in a later chunk. **This chunk trains nothing.**

**25×25 is not touched:** its data, splits and statistics stay exactly as they are. 55×55 stays excluded.

**Plan (pre-registered):**

| step | rule |
|---|---|
| 1. Universe | every antenna of the grid: the old pool + val + test (12: 7,000; 15: 6,367; 35: 4,988 + the 1,997 de-duplicated Chunk 45 extras; 45: 6,984) |
| 2. Classify | from the **true** curve. **Non-functioning** = `LABEL_MIN` false (min S11 ≥ −10 dB). **Single band** = `LABEL_MIN` true and exactly one band under the Chunk 46/47 rule (dip < −10 dB, prominence ≥ 3 dB, ≥ 150 MHz apart) |
| 3. Filter | keep single-band and non-functioning antennas; **drop** dual-band, 3+-band and functioning-with-no-resolvable-band |
| 4. De-duplicate | identical metal patterns within a grid are collapsed to one antenna, so no pattern can sit in two splits |
| 5. Sample 5,000 | target 2,500 per class; if a class has fewer, take all of it and fill the rest from the other class (seeded) |
| 6. Split | 4,000 / 500 / 500 (80/10/10), stratified by class (seeded) |

**What changes for later chunks:**
- **Old splits:** the old 12/15/35/45 splits are **not** used for training or testing in this study.
- **No leakage from earlier models:** the new test sets contain antennas that earlier models trained on. The GATv2 fine-tune must therefore restart from `best_model.pt`, which saw only 25×25. Section 8 measures that overlap.
- **Normalisation:** the contract is unchanged; targets stay in raw dB and are z-scored with the 25×25 statistics.

**Writes:**
- **Splits:** `splits/ch51_{N}x{N}_{train,val,test}.json`. Each entry is `[source, sample_idx]`, where source is `"12x12"`, `"15x15"`, `"35x35"`, `"35x35_extra"` or `"45x45"`.
- **Artifacts:** `ch51_manifest.parquet` (every antenna: class, bands, dip, selection, split), `ch51_summary.csv`, `ch51_leakage_audit.csv`, `ch51_report.md`, `figures/ch51/*`.

## 1. Installs

In [ ]:
!pip -q install torch_geometric pyarrow tabulate

## 2. Repository and Drive

In [ ]:
import os, sys, subprocess
REPO_ROOT = '/content/antenna_gnn'
if os.path.isdir(f'{REPO_ROOT}/.git'):
    subprocess.run(['git', '-C', REPO_ROOT, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/asparagusD/antenna_gnn.git', REPO_ROOT], check=True)
sys.path.insert(0, f'{REPO_ROOT}/src')
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
ART, SPLITS, LOCAL = f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/splits', '/content/local_graphs'
os.makedirs(LOCAL, exist_ok=True)
print(REPO_ROOT, DATA_ROOT)

## 3. Setup and pre-registered constants
All the rules from the table above are fixed here, before any curve is read. A CPU runtime is enough; nothing is trained.

In [ ]:
# ── Setup: pre-registered constants, provenance ──
import json, glob, hashlib, zipfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch
from scipy.signal import find_peaks
import matplotlib, matplotlib.pyplot as plt
from tqdm.auto import tqdm

GRIDS = (12, 15, 35, 45)                       # 25x25 is NOT touched; 55x55 stays excluded
F = np.linspace(1, 4, 201); TAU = -10.0
BAND = dict(height=10.0, prominence=3.0, distance=10)   # Chunk 46/47 band rule, unchanged
TARGET_N = 5000; TARGET_PER_CLASS = 2500
SPLIT_FRAC = (0.8, 0.1, 0.1)                   # -> 4,000 / 500 / 500
SEED_SAMPLE, SEED_SPLIT = 5100, 5101           # sampling and splitting seeds (fixed before any data is read)
UNIVERSE_EXPECT = {12: 7000, 15: 6367, 35: 4988, 45: 6984}; EXTRA35_EXPECT = 1997

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
READ_SHA = {}
def READ(p):
    assert os.path.exists(p), f'missing {p}'
    READ_SHA.setdefault(p, sha(p)); return p
def OUT(n):
    assert n.startswith('ch51_'), n; return f'{ART}/{n}'
def SOUT(n):
    assert n.startswith('ch51_'), n; return f'{SPLITS}/{n}'
FIG = f'{DATA_ROOT}/figures/ch51'; os.makedirs(FIG, exist_ok=True)
PRIOR = {p: os.path.getmtime(p) for pat in (f'{ART}/*', f'{SPLITS}/*') for p in glob.glob(pat)
         if os.path.isfile(p) and not Path(p).name.startswith('ch51_')}
print(f'grids {GRIDS} | target {TARGET_N:,} per grid ({TARGET_PER_CLASS:,} per class) | split {SPLIT_FRAC} | seeds {SEED_SAMPLE}/{SEED_SPLIT}')
print(f'captured mtimes of {len(PRIOR)} earlier files')

## 4. Universe of antennas
**What it does:**
- **Builds each grid's full set of antennas** from its old pool, val and test indices, using the same split files as Chunks 41–45. These are used only to enumerate the antennas and for the leakage audit.
- **35×35:** adds Chunk 45's 1,997 de-duplicated extra antennas. Each must be marked `new` in Chunk 45's manifest.
- **Checks the counts** against the known dataset sizes.

In [ ]:
# ── Universe: every antenna of each grid (old pool + val + test), plus the 35x35 extras ──
def pairs(v):
    if isinstance(v, dict):
        return {(int(g), int(i)) for g, x in v.items() for i in x}
    return {(int(e[0]), int(e[1])) for e in v}
OLD = {}
for N in (12, 15):
    OLD[N] = {r: sorted(i for g, i in pairs(json.load(open(READ(f'{SPLITS}/finetune{N}_{r}_indices.json'))))) for r in ('pool', 'val', 'test')}
FTI = {r: pairs(json.load(open(READ(f'{SPLITS}/finetune_{r}_indices.json')))) for r in ('pool', 'val', 'test')}
for N in (35, 45):
    OLD[N] = {r: sorted(i for g, i in FTI[r] if g == N) for r in ('pool', 'val', 'test')}
REP45 = json.load(open(READ(f'{ART}/ch45_preprocess_report.json')))
MAN45 = pd.read_csv(READ(f'{ART}/ch45_manifest_35x35_extra.csv'))
EXTRA35 = sorted(int(i) for i in REP45['new_pool_sample_idx'])
st = MAN45.set_index('sample_idx').status
assert len(EXTRA35) == EXTRA35_EXPECT and all(st[i] == 'new' for i in EXTRA35), 'STOP: Chunk 45 extra list'
UNIV = {}
for N in GRIDS:
    ids = sorted(set(OLD[N]['pool']) | set(OLD[N]['val']) | set(OLD[N]['test']))
    assert len(ids) == UNIVERSE_EXPECT[N], f'STOP: {N}x{N} universe {len(ids)} != {UNIVERSE_EXPECT[N]}'
    UNIV[N] = [(f'{N}x{N}', i) for i in ids] + ([('35x35_extra', i) for i in EXTRA35] if N == 35 else [])
    print(f'{N}x{N}: {len(UNIV[N]):,} antennas' + (f' ({UNIVERSE_EXPECT[N]:,} + {len(EXTRA35):,} Chunk 45 extras)' if N == 35 else ''))

## 5. Stage graphs; read every true curve and pattern
**What it does:**
- Unzips the processed graphs, including `processed_35x35_extra.zip`, to local SSD.
- Reads each antenna's raw-dB curve and N×N metal pattern from the graph: the pattern is the metal column of the pixel nodes, as in Chunks 36/42/49.
- Hashes each pattern for de-duplication.

In [ ]:
# ── Stage graphs; read true curves and patterns ──
FOLDER = {f'{N}x{N}': f'{LOCAL}/{N}x{N}' for N in GRIDS}; FOLDER['35x35_extra'] = f'{LOCAL}/35x35_extra'
ZIPS = {f'{N}x{N}': f'{DATA_ROOT}/data/processed/processed_{N}x{N}.zip' for N in GRIDS}
ZIPS['35x35_extra'] = f'{DATA_ROOT}/data/processed/processed_35x35_extra.zip'
for src, d in FOLDER.items():
    if os.path.exists(f'{d}/_STAGED.txt'):
        continue
    os.makedirs(d, exist_ok=True)
    with zipfile.ZipFile(READ(ZIPS[src])) as Z:
        mem = [m for m in Z.namelist() if m.endswith('.pt')]
        for m in tqdm(mem, desc=f'stage {src}'):
            with Z.open(m) as a, open(f'{d}/{os.path.basename(m)}', 'wb') as b:
                shutil.copyfileobj(a, b)
    open(f'{d}/_STAGED.txt', 'w').write(str(len(mem)))
Y, PAT = {}, {}
for N in GRIDS:
    ys, hs = [], []
    for src, i in tqdm(UNIV[N], desc=f'read {N}x{N}', leave=False):
        d = torch.load(f'{FOLDER[src]}/sample_{int(i)}.pt', weights_only=False)
        assert d.x.shape == (N * N + 1, 5) and float(d.x[-1, 3]) == -1.0 and d.y.numel() == 201 and float(d.y.max()) <= 0.01
        ys.append(d.y.reshape(-1).numpy().astype(np.float64))
        hs.append(hashlib.sha1(d.x[:-1, 0].numpy().astype(np.uint8).tobytes()).hexdigest())
    Y[N], PAT[N] = np.stack(ys), np.array(hs)
    print(f'{N}x{N}: {len(ys):,} curves read')

## 6. Classify, filter and de-duplicate
**What it does:**
- **Classifies** every antenna as single band, dual band, 3+ bands, non-functioning, or functioning with no resolvable band. The rules are `LABEL_MIN` and the Chunk 46/47 band rule.
- **Keeps** only single-band and non-functioning antennas.
- **De-duplicates:** collapses identical patterns within a grid. If duplicates disagree in class (they shouldn't), every copy is dropped.
- **Prints a funnel** per grid: universe → kept classes → after de-duplication.

**Writes:** the first part of `ch51_manifest.parquet`.

In [ ]:
# ── Classify every antenna, keep single-band + non-functioning, de-duplicate ──
rows = []
for N in GRIDS:
    for k, (src, i) in enumerate(UNIV[N]):
        y = Y[N][k]; pk, _ = find_peaks(-y, **BAND); nb = len(pk); func = bool(y.min() < TAU)
        if not func:
            cls = 'non_functioning'
        elif nb == 1:
            cls = 'single_band'
        elif nb == 2:
            cls = 'dual_band'
        elif nb >= 3:
            cls = 'three_plus_bands'
        else:
            cls = 'functioning_no_band'
        rows.append(dict(grid=N, source=src, sample_idx=int(i), row=k, pattern_sha1=PAT[N][k], min_db=float(y.min()),
                         argmin_ghz=float(F[y.argmin()]), n_bands=nb, label_min=func, cls=cls,
                         dip_ghz=float(F[pk[0]]) if nb == 1 else np.nan, dip_db=float(y[pk[0]]) if nb == 1 else np.nan))
MAN = pd.DataFrame(rows)
OLD_SET = {N: {r: set(OLD[N][r]) for r in ('pool', 'val', 'test')} for N in GRIDS}
def old_split(r):
    if r.source.endswith('_extra'):
        return 'pool'                                           # the Chunk 45 extras were training-only
    return next(s for s in ('pool', 'val', 'test') if r.sample_idx in OLD_SET[r.grid][s])
MAN['old_split'] = [old_split(r) for r in MAN.itertuples()]
MAN['kept_class'] = MAN.cls.isin(['single_band', 'non_functioning'])
# de-duplication within each grid, among kept antennas
MAN['dup_dropped'] = False
for N in GRIDS:
    g = MAN[(MAN.grid == N) & MAN.kept_class]
    for h, grp in g.groupby('pattern_sha1'):
        if len(grp) == 1:
            continue
        if grp.cls.nunique() > 1:
            MAN.loc[grp.index, 'dup_dropped'] = True                # conflicting labels: drop every copy
        else:
            MAN.loc[grp.index[1:], 'dup_dropped'] = True            # keep the first copy
MAN['eligible'] = MAN.kept_class & ~MAN.dup_dropped
FUN = []
for N in GRIDS:
    m = MAN[MAN.grid == N]; c = m.cls.value_counts()
    FUN.append(dict(grid=f'{N}x{N}', universe=len(m), single_band=int(c.get('single_band', 0)), non_functioning=int(c.get('non_functioning', 0)),
                    dual_band=int(c.get('dual_band', 0)), three_plus=int(c.get('three_plus_bands', 0)), func_no_band=int(c.get('functioning_no_band', 0)),
                    kept=int(m.kept_class.sum()), duplicates_dropped=int(m.dup_dropped.sum()),
                    eligible_single=int((m.eligible & (m.cls == 'single_band')).sum()),
                    eligible_nonfunc=int((m.eligible & (m.cls == 'non_functioning')).sum())))
FUN = pd.DataFrame(FUN); print(FUN.to_string(index=False))

## 7. Sample 5,000 per grid and split 4,000 / 500 / 500
**Sampling (seed 5100):**
- Take 2,500 per class where available.
- If a class has fewer than 2,500 eligible antennas, take all of it and fill the remainder from the other class, so every grid has exactly 5,000.
- 35/45 are expected to land at about 50.7% / 52% single-band.

**Splitting (seed 5101), stratified by class:** 80% / 10% / 10% of each class, rounded so the splits are exactly 4,000 / 500 / 500.

**Writes:** `splits/ch51_{N}x{N}_{train,val,test}.json` and the completed `ch51_manifest.parquet`.

In [ ]:
# ── Sample 5,000 per grid (about 50/50), stratified 4,000 / 500 / 500 split ──
def alloc(n_total, sizes):
    # split n_total over classes in proportion to sizes, rounding so the parts sum exactly to n_total
    raw = np.array(sizes, float) / sum(sizes) * n_total; base = np.floor(raw).astype(int)
    base[np.argsort(-(raw - base))[: n_total - base.sum()]] += 1; return base
MAN['selected'] = False; MAN['split'] = ''
for N in GRIDS:
    rng = np.random.default_rng(SEED_SAMPLE + N)
    el = {c: MAN.index[(MAN.grid == N) & MAN.eligible & (MAN.cls == c)].to_numpy() for c in ('single_band', 'non_functioning')}
    short = {c: max(0, TARGET_PER_CLASS - len(v)) for c, v in el.items()}
    take = {'single_band': TARGET_PER_CLASS + short['non_functioning'] - short['single_band'],
            'non_functioning': TARGET_PER_CLASS + short['single_band'] - short['non_functioning']}
    take = {c: min(n, len(el[c])) for c, n in take.items()}
    assert sum(take.values()) == TARGET_N, f'STOP: {N}x{N} cannot reach {TARGET_N} eligible antennas ({take})'
    chosen = {c: np.sort(rng.choice(el[c], take[c], replace=False)) for c in el}
    rs = np.random.default_rng(SEED_SPLIT + N)
    n_split = {'train': int(TARGET_N * SPLIT_FRAC[0]), 'val': int(TARGET_N * SPLIT_FRAC[1])}; n_split['test'] = TARGET_N - n_split['train'] - n_split['val']
    per_cls = {c: dict(zip(('train', 'val', 'test'), alloc(len(chosen[c]), [SPLIT_FRAC[0], SPLIT_FRAC[1], SPLIT_FRAC[2]]))) for c in chosen}
    # fix rounding so the split totals are exactly 4,000 / 500 / 500
    for s in ('val', 'test'):
        diff = n_split[s] - sum(per_cls[c][s] for c in chosen)
        if diff:
            c = max(chosen, key=lambda c: per_cls[c]['train']); per_cls[c][s] += diff; per_cls[c]['train'] -= diff
    for c, idx in chosen.items():
        perm = rs.permutation(idx); a, b = per_cls[c]['train'], per_cls[c]['train'] + per_cls[c]['val']
        for s, part in (('train', perm[:a]), ('val', perm[a:b]), ('test', perm[b:])):
            MAN.loc[part, 'selected'] = True; MAN.loc[part, 'split'] = s
    for s in ('train', 'val', 'test'):
        sel = MAN[(MAN.grid == N) & (MAN.split == s)].sort_values(['source', 'sample_idx'])
        assert len(sel) == n_split[s], (N, s, len(sel))
        json.dump([[r.source, int(r.sample_idx)] for r in sel.itertuples()], open(SOUT(f'ch51_{N}x{N}_{s}.json'), 'w'))
    print(f'{N}x{N}: selected single {take["single_band"]:,} / non-functioning {take["non_functioning"]:,} '
          f'({take["single_band"] / TARGET_N:.1%} single) -> ' + ', '.join(f'{s} {n_split[s]:,}' for s in n_split))
MAN.to_parquet(OUT('ch51_manifest.parquet'), index=False)

## 8. Summary and leakage audit
**Summary:** per grid and split, the counts per class, the single-band share, and the resonance-frequency and dip-depth statistics of the single-band antennas. These show whether the three splits are alike.

**Leakage audit:** how many new val/test antennas were in the **old training pools**. Every earlier 12/15/35/45 model (`backbone_no55` onward) trained on those pools, so any overlap above zero means fine-tuning must restart from `best_model.pt`.

**Writes:** `ch51_summary.csv`, `ch51_leakage_audit.csv`.

In [ ]:
# ── Summary per grid and split; leakage audit against the old training pools ──
SUM = []
for N in GRIDS:
    for s in ('train', 'val', 'test'):
        m = MAN[(MAN.grid == N) & (MAN.split == s)]; sb = m[m.cls == 'single_band']
        SUM.append(dict(grid=f'{N}x{N}', split=s, n=len(m), single_band=int((m.cls == 'single_band').sum()),
                        non_functioning=int((m.cls == 'non_functioning').sum()), single_share=(m.cls == 'single_band').mean(),
                        from_extra=int(m.source.str.endswith('_extra').sum()),
                        dip_ghz_median=sb.dip_ghz.median(), dip_ghz_iqr=sb.dip_ghz.quantile(.75) - sb.dip_ghz.quantile(.25),
                        dip_ghz_p5=sb.dip_ghz.quantile(.05), dip_ghz_p95=sb.dip_ghz.quantile(.95),
                        dip_db_median=sb.dip_db.median(), min_db_median_nonfunc=m[m.cls == 'non_functioning'].min_db.median()))
SUM = pd.DataFrame(SUM); SUM.to_csv(OUT('ch51_summary.csv'), index=False); print(SUM.round(3).to_string(index=False))
LEAK = []
for N in GRIDS:
    for s in ('val', 'test'):
        m = MAN[(MAN.grid == N) & (MAN.split == s)]
        LEAK.append(dict(grid=f'{N}x{N}', new_split=s, n=len(m), were_in_old_training_pool=int((m.old_split == 'pool').sum()),
                         were_in_old_val=int((m.old_split == 'val').sum()), were_in_old_test=int((m.old_split == 'test').sum())))
LEAK = pd.DataFrame(LEAK); LEAK.to_csv(OUT('ch51_leakage_audit.csv'), index=False)
print('\nLeakage audit (new val/test antennas that earlier models trained on):'); print(LEAK.to_string(index=False))
if LEAK.were_in_old_training_pool.sum() > 0:
    print('\n=> Earlier 12/15/35/45 models saw part of the new val/test sets. The fine-tune chain MUST restart from best_model.pt (25x25 only).')

## 9. Figures and report
**Figures:**
- class composition per grid and split
- resonance-frequency histograms of the single-band antennas per split, which should overlap

**Writes:** `figures/ch51/ch51_composition.png`, `figures/ch51/ch51_dip_frequency.png` and `ch51_report.md`, which holds the plan, the funnel, the split summary, the leakage audit and instructions for the next chunk.

In [ ]:
# ── Figures and report ──
fig, ax = plt.subplots(figsize=(7, 3))
lab, sb, nf = [], [], []
for N in GRIDS:
    for s in ('train', 'val', 'test'):
        r = SUM[(SUM.grid == f'{N}x{N}') & (SUM.split == s)].iloc[0]; lab.append(f'{N}\n{s}'); sb.append(r.single_band); nf.append(r.non_functioning)
x = np.arange(len(lab)); ax.bar(x, sb, color='#0072B2', label='single band'); ax.bar(x, nf, bottom=sb, color='#BBBBBB', label='non-functioning')
ax.set_xticks(x); ax.set_xticklabels(lab, fontsize=7); ax.set_ylabel('antennas'); ax.set_yscale('log'); ax.legend(frameon=False, fontsize=8)
ax.set_title('Chunk 51 composition per grid and split'); fig.tight_layout(); fig.savefig(f'{FIG}/ch51_composition.png', dpi=200); plt.show(); plt.close(fig)
fig, axs = plt.subplots(1, len(GRIDS), figsize=(12, 2.6), sharey=False)
for ax, N in zip(axs, GRIDS):
    for s, c in (('train', '#0072B2'), ('val', '#E69F00'), ('test', '#009E73')):
        v = MAN[(MAN.grid == N) & (MAN.split == s) & (MAN.cls == 'single_band')].dip_ghz
        ax.hist(v, bins=np.arange(1, 4.03, 0.06), density=True, histtype='step', color=c, label=s)
    ax.set_title(f'{N}x{N}'); ax.set_xlabel('single-band dip (GHz)')
axs[0].legend(frameon=False, fontsize=7); fig.tight_layout(); fig.savefig(f'{FIG}/ch51_dip_frequency.png', dpi=200); plt.show(); plt.close(fig)

L = ['# Chunk 51 — Balanced single-band datasets (12/15/35/45); 25×25 untouched', '',
     'Classes from the true curve: non-functioning = min S11 ≥ −10 dB (`LABEL_MIN` false); single band = `LABEL_MIN` true and exactly one '
     'band (dip < −10 dB, prominence ≥ 3 dB, ≥ 150 MHz apart). Dual-band, 3+-band and functioning-with-no-resolvable-band antennas are excluded. '
     'Identical patterns collapsed. 5,000 per grid, ≈50/50, split 4,000 / 500 / 500 stratified by class (seeds 5100 / 5101).', '',
     '## Funnel', FUN.to_markdown(index=False), '', '## Splits', SUM.round(3).to_markdown(index=False), '',
     '## Leakage audit (new val/test antennas that were in the old training pools)', LEAK.to_markdown(index=False), '',
     '## For the training chunk',
     '- Load `splits/ch51_{N}x{N}_{train,val,test}.json`; each entry is `[source, sample_idx]` (source `35x35_extra` = `processed_35x35_extra.zip`).',
     '- GATv2: restart from `best_model.pt` (25×25 only; 5 → 6 inputs with a zero column), fine-tune on the new 12/15/35/45 train sets with '
     '25×25 replay from the unchanged `indices.json` train split; select on the new val sets.',
     '- Gupta CNN: train from scratch per grid on the new train sets, select on the new val sets.',
     '- Normalisation contract unchanged (25×25 `s11_mean` / `s11_std`). Score the new test sets once.',
     '- Results are for single-band and non-functioning antennas only, and are not comparable with the natural-distribution table.']
open(OUT('ch51_report.md'), 'w').write('\n'.join(L) + '\n'); print('\n'.join(L))

## 10. Guards
**What it checks:**
- every grid has exactly 5,000 antennas, split 4,000 / 500 / 500, with no antenna in two splits
- every selected antenna is single band or non-functioning, and eligible
- no metal pattern appears in two splits
- the single-band share is between 45% and 55%
- the 35×35 extras come only from Chunk 45's de-duplicated list
- the split files reload exactly
- 25×25 files are untouched, input files unchanged, no earlier file modified
- every output named `ch51_*`

In [ ]:
# ── Guards ──
fails = []
def guard(ok, msg):
    print(f"  [{'PASS' if ok else 'FAIL'}] {msg}")
    if not ok: fails.append(msg)
for N in GRIDS:
    m = MAN[(MAN.grid == N) & MAN.selected]
    guard(len(m) == TARGET_N and dict(m.split.value_counts()) == {'train': 4000, 'val': 500, 'test': 500}, f'{N}x{N}: 5,000 = 4,000 / 500 / 500')
    guard(m.cls.isin(['single_band', 'non_functioning']).all() and m.eligible.all(), f'{N}x{N}: only eligible single-band / non-functioning')
    guard(not m.duplicated(['source', 'sample_idx']).any(), f'{N}x{N}: no antenna selected twice')
    guard(m.groupby('pattern_sha1').split.nunique().max() == 1, f'{N}x{N}: no pattern in two splits')
    share = (m.cls == 'single_band').mean(); guard(0.45 <= share <= 0.55, f'{N}x{N}: single-band share {share:.1%} within 45-55%')
    for s in ('train', 'val', 'test'):
        J = json.load(open(SOUT(f'ch51_{N}x{N}_{s}.json')))
        guard(sorted(map(tuple, J)) == sorted(zip(m[m.split == s].source, m[m.split == s].sample_idx.astype(int))), f'{N}x{N} {s}: split file reloads exactly')
ex = MAN[MAN.selected & (MAN.source == '35x35_extra')]
guard(set(ex.sample_idx) <= set(EXTRA35), '35x35 extras only from the Chunk 45 de-duplicated list')
guard(not any('25x25' in p for p in glob.glob(f'{SPLITS}/ch51_*') + glob.glob(f'{ART}/ch51_*')), '25x25 untouched (no ch51 file for it)')
guard(all(sha(p) == h for p, h in READ_SHA.items()), f'input files unchanged ({len(READ_SHA)})')
guard(all(os.path.getmtime(p) == t for p, t in PRIOR.items() if os.path.exists(p)), f'no earlier file modified ({len(PRIOR)})')
guard(all(Path(p).name.startswith('ch51_') for p in glob.glob(f'{ART}/ch51_*') + glob.glob(f'{SPLITS}/ch51_*')), 'all outputs ch51_*')
assert not fails, fails
print('ALL GUARDS PASSED. Data prepared; nothing trained.')